# Gold Layer: Equipment Health Dashboard

**Purpose:** Create business-level metric showing overall equipment health

**Why this table exists:**
- Plant manager need ONE view of equipment status
- Instead of raw sensor readings, they see "healthy" or "at-risk"
- Enables quick decision making (which equipment to service?)

**Input:** `dev.silver.sensor_readings_enriched`
(clean, deduplicated sensor data with equipment details)

**Output:** `dev.gold.equipment_health_dashboard`
(one row per equipment with health score)

**Example Output:**
| Equipment ID | Equipment Type | Factory | Health Score | Risk Level | Last Updated |
| ------------ | -------------- | ------- | ------------ | ---------- | ------------ |
| EQ-0001      | CNC Machine    | Factory_North | 82     | LOW        | 2024-01-15 14:30 |
| EQ-0002      | Conveyor       | Factory_South | 35     | HIGH       | 2024-01-15 14:30 |

**Business Value:**
- Dashboard shows which equipment needs attention
- Helps prioritize maintenance
- Reduces unexpected failures

## Step 1: Configuration

Define table names and thresholds

In [0]:
# Congiguration: Define where data comes from and goes to

# INPUT: Where we read clean sensor data
INPUT_TABLE = "dev.silver.sensor_readings_enriched"

# OUTPUT: Where we write business matrics
OUTPUT_TABLE = "dev.gold.equipment_health_dashboard"

# HEALTH SCORE THRESHOLDS
# Why these threasholds?
# - Based on typical sensor ranges from our data
# - Temperature: normal 40-80°C, alert if avg > 75°C
# - Vibration: normal 0.01-0.05, alert if avg > 0.08
# - Pressure: normal 80-120 PSI, alert if avg > 130
TEMP_THRESHOLD = 75 # Celsius - alert if higher
VIBRATION_THRESHOLD = 0.08 #nm/s - alert if higher
PRESSURE_THRESHOLD = 130 # PSI - alert if higher

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Input table: {INPUT_TABLE}")
print(f"Output table: {OUTPUT_TABLE}")
print(f"\nHealth threasholds:")
print(f"Temperature: > {TEMP_THRESHOLD}°C = Alert")
print(f"Vibration: > {VIBRATION_THRESHOLD} mm/s = Alert")
print(f"Pressure: > {PRESSURE_THRESHOLD} PSI = Alert")

## Step 2: Load Input Data

Read and enriched sensor data from Silver layer

In [0]:
# Load enriched sensor data
# Why enriched data? Because it has BOTH:
# - Sensor readings (temperature, vibration, etc.)
# - Equipment details (equipment_type, factory, criticality)
# This lets us create meaningful business matrics

enriched_df = spark.read.table(INPUT_TABLE)

print(f"Loaded {enriched_df.count()} sensor readings")
print(f"\nData preview (showing key columns):")
enriched_df.select(
    "equipment_id",
    "equipment_type",
    "sensor_type",
    "value",
    "factory_location",
    "equipment_criticality"
).show(10, truncate=False)

print(f"\nUnique equipment: {enriched_df.select('equipment_id').distinct().count()}")
print(f"Unique sensor types: {enriched_df.select('sensor_type').distinct().count()}")

## Setp 3: Calculate Average Sensor Values Per Equipment

Group by equipment and calculated statistics

**WHY?** We need to know typical sensor values for each machine
- If EQ-0001's average temperature is 75°C, that's a warning sign
- If EQ-0001's average temperature is 62°C, that's normal

In [0]:
from pyspark.sql.functions import when, avg, count, col, round as spark_round, max as spark_max, min as spark_min

# GROUP BY equipment_id and calculate statistics per equipment
# Why group by equipment?
# - We want ONE health score per equipment (not per sensor)
# - So we aggregate all sensors for each equipment

equipment_stats = enriched_df \
    .groupBy(
        "equipment_id",
        "equipment_name",
        "equipment_type",
        "factory_location",
        "equipment_criticality"
    ) \
        .agg(
            # Count total readings - show how active the equipment is 
            count("*").alias("total_readings"),

            # Temperature stats - helps detect overheading issues
            spark_round(avg(
                when(col("sensor_type") == "temperature", col("value"))
            ),4).alias("avg_temperature"),

            # Vibration stats - helps detect mechanical issue
            spark_round(avg(
                when(col("sensor_type") == "vibration", col("value"))
            ),4).alias("avg_vibration"),

            # Pressure stats - helps detect hydraulic issue
            spark_round(avg(
                when(col("sensor_type") == "pressure", col("value"))
            ), 2).alias("avg_pressure"),

            # Power consumption - helps detect electrical issues
            spark_round(avg(
                when(col("sensor_type") == "power_consumption", col("value"))
            ),2).alias("avg_power")
        )

print(f"Calculated stats for {equipment_stats.count()} equipment")
print(f"\nSample equipment stats:")
equipment_stats.show(5, truncate=False)

## Step 4: Create Health Score

Calculate a single health score (0-100) for each equipment

**HOW HEALTH SCORE WORKS:**

Start with 100points (perfect health)
└─ Deduct points for each problem:
   ├─ High temerature: -20 points
   ├─ High vibration: -15 points
   ├─ High pressure: -15 points
   ├─ Low readings: -10 points (equipment not running)
   ├─ Minimum score: 0 (completely broke)


**EXAMPLES:**
- All sensors normal: 100 points  = HEALTHY
- 1 warning (high temp): 80 points = CAUTION
- 2 warnings (temp + vibration) : 65 points = AT RISK
- 3+ warnings: 30-50 = CRITICAL
   

In [0]:
# Create health score by deducting points for problems
# Health score helpts business quickly understand equipment status
# Instead of looking at 10 sensor values, they see ONE number

health_scored = equipment_stats.withColumn(
    "health_score",
    (
        # Start with 100 points (perfect health)
        100

        # Deduct 20 points if temerature is hhigh
        # Why temerature? High causes equipment failure
        - when(col('avg_temperature') > TEMP_THRESHOLD, 20).otherwise(0)

        # Deduct 15 points if viration  is high
        # Why vibration? High vibration indicates mechanical wear
        - when(col('avg_vibration') > VIBRATION_THRESHOLD, 15).otherwise(0)

        # Deduct 15 points if pressure is high
        # Why pressure? High pressure dameges seals and values
        - when(col('avg_pressure') > PRESSURE_THRESHOLD, 15).otherwise(0)

        # Deduct 10 points if very few readings (equipment not running much)
        # Why readings count? if no readings can't monitor the equipment
        - when(col('total_readings') < 10, 10).otherwise(0)
    )
).withColumn(
    # Make sure score stays between 0 and 100
    "health_score",
    when(col("health_score") < 0, 0) #Minimum : 0
    .otherwise(col("health_score"))
)

print(f"Health score calculated")
print(f"\n Health score distribution:")
health_scored.select("equipment_id", "health_score", "avg_temperature", "avg_vibration").show(10)

## Step 5: Classify Risk Level

Convert heatlh score to business-friendly risk leavel

**RISK LEVEL MAPPING:**
- 80-100: LOW (green - all good)
- 60-79: MEDIUM (yellow - watch)
- 40-59: HIGH (orange - act soon)
- 0-39: CRITICAL (red - act now!)

In [0]:
# Classify risk level based on health score
# Why risk levels? Easier for non-technical people to understand
# "Health score 42" means nothing to a plant manager 
# "Risk level HIGH" means "we need to do something about this"

risk_classified = health_scored.withColumn(
    "risk_level",

    # Use IF-THEN logic to classify
    when(col("health_score") >= 80, "LOW")
    .when(col("health_score") >= 60, "MEDIUM")
    .when(col("health_score") >= 40, "HIGH")
    .otherwise("CRITICAL")
)

print(f"Risk levels assigned")
print(f"\nRisk level ditribution:")
#Count how many equipment in each risk category
risk_distribution = risk_classified.groupBy("risk_level").count().orderBy("count", ascending=False)
risk_distribution.show()

print(f"Sample with risk levels:")
risk_classified.select(
    "equipment_id",
    "equipment_type",
    "health_score",
    "risk_level",
    "avg_temperature",
    "avg_vibration"
).show(10, truncate=False)


## Step 6: Add Timestamp

Record when this health assessment was calculated

**WHY?** Dashboard users need to know how fresh the data is
- "Last updated 2 minutes ago" = good, can act on it
- "Last updated yesterday" = old, data might be stale

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Add timestamp of when this dashboard was calculated
# Plant managers see "Last Updated:2024-01-15 14:30:45"
# They know if data is current or outdated

final_dashboard = risk_classified.withColumn(
    "dashboard_timestamp",
    current_timestamp() # Current date and time
).withColumn(
    "dashboard_version",
    lit("1.0")
)

print(f"Timestamp added")
print(f"Final dashboard (ready for business):")
final_dashboard.select(
    "equipment_id",
    "equipment_type",
    "factory_location",
    "health_score",
    "risk_level",
    "equipment_criticality",
    "dashboard_timestamp"
).show(10, truncate=False)

## Step 7: Reorder Columns for Dashboard

Put most important columns first (for ease of reading)

In [0]:
# Reorder columns: most important first
# Whey recorder? When plant managers open dashboard,
# they see KEY information immediately:
# - Which equipment? (equipment_id)
# - What type? (equipment_type)
# - How healthy? (health_score)
# - What risk? (risk_level)
# Then detailed metrics come after

dashboard_final = final_dashboard.select(
    # PRIORITY 1: What equipment and status?
    "equipment_id",
    "equipment_name",
    "equipment_type",
    "health_score",
    "risk_level",

    # PRIORITY 2: Where and how important 
    "factory_location",
    "equipment_criticality",

    # PRIORITY 3: Detailed sensor metrics
    "avg_temperature",
    "avg_vibration",
    "avg_pressure",
    "avg_power",

    # PRIORITY 4: Metadata
    "total_readings",
    "dashboard_timestamp",
    "dashboard_version"
)

print(f"Columns reordered for dashboard view")
print(f"\nfinal schema (dashboard columns in order)")
dashboard_final.printSchema()

## Step 8: Write to Gold Table

Save dashboard to Delta table for BI tools to consume

In [0]:
# Write to Gold layer
# Why Gold? This is the final "business-ready" table
# BI tools (Tableau, Power BI) will read form this table
# Plant managers will see this in their dashboards

print(f"Writing {dashboard_final.count()} equipment to {OUTPUT_TABLE}...")

dashboard_final.write \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(OUTPUT_TABLE)

print(f"Gold dashboard table written!")
print(f"Table: {OUTPUT_TABLE}")
print(f"Records: {dashboard_final.count()}")

## Step 9: Verify Dashboard Quality

Read back and check the dashboard looks correct

In [0]:
# Read back the dashboard to verify it's correct
dashboard_verify = spark.read.table(OUTPUT_TABLE)

print("=" * 80)
print("EQUIPMENT HEALTH DASHBOARD (FINAL)")
print("=" * 80)

# Show full dashboard with all important columns
print("\nFull Dashboard View:")
dashboard_verify.show(20, truncate=False)

print(f"\nDashboard Summary:")
print(f"Total equipment: {dashboard_verify.count()}")

## Step 10: Dashboard Business Metrics

show metrics that matter to business stakeholders

In [0]:
# Show metrics that plant managers care about
# This is what they'll use to make decisions

print("=" * 80)
print("BUSINESS DASHBOARD METRICS")
print("=" * 80)

# Count equipment by risk level
print("\nEQUIPMENT DISTRIBUTIN BY RISK:")
risk_summary = dashboard_verify.groupBy("risk_level").count().orderBy(
    when(col("risk_level") == "CRITICAL", 0)
    .when(col("risk_level") == "HIGH", 1)
    .when(col("risk_level") == "MEDIUM", 2)
)
risk_summary.show()

# Which factories have most issues?
print("\n RISK BY FACTORY:")
#Why show by factory? Plant managers can see which factory needs attention
dashboard_verify.groupBy("factory_location").agg(
    count("*").alias("total_equipment"),
    # Count how many are HIGH OR CRITICAL risk
    count(when(
        (col("risk_level") == "HIGH") | (col("risk_level") == "CRITICAL"), 
        True
    )).alias("at_risk_equipment ")
).show()

# Average health score by equipment type
print("HEALTH BY EQUIPMENT TYPE:")
# Why by type?  Different equipment has different failure modes
dashboard_verify.groupBy("equipment_type").agg(
    spark_round(avg("health_score"), 2).alias("avg_health_score"), 
    count("*").alias("count")
).orderBy("avg_health_score").show()

# Which specific equipment needs immediate attention?
print("\nEQUIPMENT NEEDING IMMEDIATE ACTION (CRITICAL RISK):")
# These are the ones plantmanagers should look at TODAY
dashboard_verify.filter(col("risk_level") == "CRITICAL") \
    .select("equipment_id", "equipment_name", "health_score", "avg_temperature", "avg_vibration") \
    .orderBy("health_score") \
    .show(10, truncate=False)

print("\n" + "=" * 80)

## Step 11: Summary Report

Complete overview of what we build

In [0]:
# Final summary showing what the dashboard provides

print("=" * 80)
print("GOLD LAYER: EQUIPMENT HEALTH DASHBOARD - COMPLETE!")
print("=" * 80)

print(f"""
      WHAT WE BUILD:
      A single-row-per-equipment dashboard showing health status

      BUSINESS  VALUE:
      - Plant managers see equipment status at a glance
      - Health score (0-100) is easy to understand
      - Risk levels (LOW/MEDIUM/HIGH/CRITICAL) drive action
      - Identify which equipment to service

      DATA SOURCE:
      Input: {INPUT_TABLE} (cleann deduplicated, enriched)
      Output: {OUTPUT_TABLE} (business dashboard)

      KEY METRICS CALCULATED:
      - HEALTH SCHORE: 0-100 (lower = more problems)
      - Risk Level: LOW/MEDIUM/HIGH/CRITICAL
      - Average Sensor Values: Temerature, Vibration, Pressure, Power
      - Reading Count: How active each equipment is

      DECISON SUPPORT
      Equipment with CRITICAL risk -> Service TODAY
      Equipment with HIGH risk -> Service this week
      Equipment with MEDIUM risk -> Monitor closely
      Equipment with LOW risk -> Normal operation

      HOW IT'S USED:
      1. BI Tools (Tableau/Power BI) read this table
      2. Dashboard shows equipment status in real-time
      3. Plant managers see red/yellow/green indicators
      4. They prioritize maintenance work
      5. Reduces unexpected downtime
      """)

print("=" * 80)